**hwpapi**는 한컴 오피스(아래아 한글)의 `HwpObject` COM 인터페이스를 감싸 파이썬에서 **한컴 오피스 자동화**를 쉽게 할 수 있도록 만든 라이브러리입니다.

xlwings가 Excel을 파이썬으로 다루는 것을 쉽게 해주듯, hwpapi는 HWP 문서 작업을 간결하고 직관적으로 만들어 줍니다.

- 📖 [Tutorials](tutorials/01_app_basics.html)
- 🧪 [API Reference](api/00_core.html)
- 💻 [GitHub 저장소](https://github.com/JunDamin/hwpapi)

## 주요 특징

| 영역 | 기능 |
|------|------|
| **텍스트** | `insert_text`, `replace_all`, `find_text`, `get_text`, `select_text` |
| **서식** | `set_charshape`, `set_parashape`, `get_charshape`, `get_parashape` |
| **탐색** | `app.move.top_of_file()`, `bottom_of_file()`, `current_list()` 등 |
| **파일** | `save`, `open`, `close`, 포맷 자동 감지 |
| **파라미터셋** | 704개 액션 · 286개 ParameterSet · 자동 등록된 property descriptor |
| **단위 변환** | `"210mm"`, `"21cm"`, `"8.27in"`, `"12pt"` ↔ HWPUNIT 자동 변환 |
| **COM 직접 접근** | `app.api` — win32com의 모든 기능을 그대로 사용 가능 |

## 설치

```sh
pip install hwpapi
```

::: {.callout-important}
이 패키지는 `win32com`을 통해 한글 오피스를 제어하므로 **한글 오피스가 설치된 Windows**에서만 작동합니다.
Linux나 macOS에서는 작동하지 않습니다.
:::

## Quick Start

```python
from hwpapi.core import App

# 한글 실행 (이미 실행 중이면 연결)
app = App(is_visible=True)

# 텍스트 입력
app.insert_text("안녕하세요, hwpapi!\n")

# 첫 줄을 선택하고 제목 서식 적용
app.move.top_of_file()
app.select_text()
app.set_charshape(bold=True, height=2000, text_color="#FF0000")

# 찾기 / 바꾸기
app.replace_all("hwpapi", "HWPAPI")

# 저장
app.save("hello.hwp")
```

## 기존 win32com 코드와 비교

기존 `win32com`으로 한글 자동화를 하면 다음과 같이 코드가 장황합니다.

```python
import win32com.client as win32
hwp = win32.gencache.EnsureDispatch("HWPFrame.HwpObject")
hwp.XHwpWindows.Item(0).Visible = True

act = hwp.CreateAction("InsertText")
pset = act.CreateSet()
pset.SetItem("Text", "Hello\r\nWorld!")
act.Execute(pset)
```

hwpapi를 쓰면 한 줄이면 됩니다.

```python
from hwpapi.core import App
app = App()
app.insert_text("Hello\r\nWorld!")
```

글자 모양 변경도 마찬가지입니다. 기존 방식:

```python
Act = hwp.CreateAction("CharShape")
Set = Act.CreateSet()
Act.GetDefault(Set)
Set.SetItem("Italic", 1)
Act.Execute(Set)
```

hwpapi 방식:

```python
app.set_charshape(italic=True)
```

## 기능 둘러보기

### 1. 문자 서식 — 707개 속성 중 자주 쓰이는 것

```python
# 굵게·기울임·밑줄·취소선·위첨자 등 모두 지원
app.set_charshape(
    bold=True,
    italic=True,
    underline_type=1,       # 1=single, 2=double
    strike_out_type=1,      # 취소선
    super_script=1,         # 위첨자
    text_color="#FF0000",   # 빨강
    shade_color="#FFFF00",  # 형광펜 노랑
    height=2400,            # 24pt
    spacing_hangul=50,      # 자간
)

# 현재 커서 위치의 서식 읽기
cs = app.get_charshape()
print(cs.Bold, cs.Height, cs.TextColor)
```

### 2. 문단 서식 — 정렬·줄간격·들여쓰기

```python
app.set_parashape(
    align_type=3,      # 1=left, 2=right, 3=center, 4=justify
    line_spacing=200,  # 200% 줄간격
    indentation=2000,  # 첫줄 들여쓰기 (HWPUNIT)
    left_margin=100,
)

pa = app.get_parashape()
print(pa.AlignType, pa.LineSpacing)
```

### 3. 단위 변환 — HWPUNIT 몰라도 OK

HWP 내부 단위인 HWPUNIT (1mm ≈ 283) 대신 익숙한 단위를 그대로 씁니다.

```python
from hwpapi.parametersets import PageDef

page = PageDef()
page.width  = "210mm"   # A4 너비
page.height = "297mm"   # A4 높이
page.width  = "21cm"    # 동일
page.width  = "8.27in"  # 동일
page.width  = 210       # 숫자만 쓰면 기본 단위(mm)
```

### 4. 찾기 · 바꾸기

```python
app.find_text("검색어")                   # 첫 번째 발견 위치로 이동
count = app.replace_all("before", "after")  # 전체 치환, 개수 반환

# 서식 조건으로 찾기
action = app.actions.ForwardFind
action.pset.FindString = "중요"
action.pset.find_char_shape.bold = True  # 굵은 글씨만
action.run()
```

### 5. 표 만들고 채우기

```python
action = app.actions.TableCreate
action.pset.Rows = 3
action.pset.Cols = 4
action.run()

for i in range(12):
    app.insert_text(f"셀 {i+1}")
    app.api.Run("TableRightCell")
```

### 6. 704개 액션에 접근

한글의 모든 편집 기능은 "액션"으로 노출됩니다. hwpapi는 704개 액션을 자동으로 등록합니다.

```python
# 액션 목록 조회
app.actions.list_actions()                    # 전체
app.actions.list_actions(with_pset_only=True) # pset을 받는 것만

# 임의 액션 실행
app.actions.Undo.run()
app.actions.SelectAll.run()
app.actions.Copy.run()
app.actions.Paste.run()

# pset이 있는 액션은 파라미터를 먼저 설정
a = app.actions.InsertHyperlink
a.pset.Text = "링크 텍스트"
a.pset.Command = "https://example.com"
a.run()
```

### 7. 언제든 COM으로 내려가기

hwpapi가 감싸지 못한 기능은 `app.api`를 통해 `win32com`의 모든 API를 그대로 쓸 수 있습니다.

```python
# 원시 HwpObject 직접 접근
app.api.XHwpDocuments.Item(0).Save()
app.api.Run("BreakPara")

# HParameterSet 직접 조작
app.api.HAction.GetDefault("CharShape", app.api.HParameterSet.HCharShape.HSet)
app.api.HParameterSet.HCharShape.Bold = 1
app.api.HAction.Execute("CharShape", app.api.HParameterSet.HCharShape.HSet)
```

## ParameterSet 시스템

286개 ParameterSet이 도메인별로 정리되어 있습니다. 모든 속성은 property descriptor로 자동 변환·검증·단위 변환을 수행합니다.

```
hwpapi/parametersets/
├── __init__.py     # 기반
├── mappings.py     # 35개 문자열↔숫자 매핑 (ALIGN_MAP, DIRECTION_MAP, ...)
├── backends.py     # 4개 백엔드 (Pset/HParam/Com/Attr)
├── properties.py   # IntProperty, BoolProperty, ColorProperty, UnitProperty, ...
├── base.py         # ParameterSet 기반 클래스, 메타클래스
└── sets/           # 143개 서브클래스 (10개 파일로 분리)
    ├── primitives.py      # CharShape, ParaShape, BorderFill, ...
    ├── text.py            # 문자 조작
    ├── table.py           # Table, CellBorderFill, ...
    ├── drawing.py         # ShapeObject, Draw*
    ├── document.py        # DocumentInfo, PageDef, SecDef, ...
    ├── file_ops.py        # FileOpen, Print, ...
    ├── find_edit.py       # FindReplace, BookMark, ...
    └── media_misc.py      # OleCreation, HyperLink, ...
```

자세한 내용은 [ParameterSet 아키텍처 문서](https://github.com/JunDamin/hwpapi/blob/main/docs/PARAMETERSET_ARCHITECTURE.md)를 참고하세요.

## 테스트 현황

- **2,306개 단위 테스트** 통과 (HWP 없이 실행 가능)
- **13개 End-to-End 시나리오** (`tests/smoke_scenarios.py`) — 실제 HWP에서 동작 검증
- **18개 Feature 테스트** (`tests/smoke_features.py`) — 문자 서식, 문단 서식, 탭, 복사/붙여넣기, Undo/Redo, 날짜 필드, 책갈피, 하이퍼링크 등
- 각 시나리오는 `GetDefault`로 HWP 상태를 **읽어서 assertion**으로 검증합니다.

```sh
# 단위 테스트
python -m pytest tests/ -q

# HWP가 필요한 E2E 테스트
python tests/smoke_scenarios.py
python tests/smoke_features.py
```

## 왜 hwpapi를 만들었나요?

가장 큰 이유는 **스스로 사용하기 위해서**입니다. 직장인으로 많은 한글 문서를 편집·작성하면서 단순 반복 업무가 너무 많다는 것이 항상 불만이었습니다.

파이콘에서 한글 자동화 발표를 본 것이 계기였고, 특히 '[회사원 코딩](https://employeecoding.tistory.com/72)' 블로그와 영상을 많이 참조했습니다. 그 과정에서 설명 자료가 부족하다는 점, 예전 코드를 계속 찾아보게 된다는 점이 불편해서 *아래아 한글용 파이썬 패키지가 있으면 좋겠다* 는 생각을 하게 되었습니다.

업무에서 엑셀 자동화를 위해 `xlwings`를 쓰면서 파이써닉한 래퍼가 작업 효율을 얼마나 올려주는지 체감했고, 그 경험을 HWP에도 가져오고 싶었습니다.

**설계 철학**은 xlwings를 따릅니다:

1. **자주 쓰는 것은 쉽게** — `insert_text`, `set_charshape` 같은 메소드로 한 줄 API 제공
2. **부족한 것은 열어두기** — `app.api`로 `win32com`의 모든 기능 접근 가능
3. **타입 안전성** — property descriptor로 자동 변환·검증
4. **Pythonic** — HWPUNIT 대신 `"210mm"`, BBGGRR 대신 `"#FF0000"`

이런 형태의 작업을 통해서 HWP API 래퍼가 활성화되어 많은 분들이 단순 작업을 자동화 할 수 있기를 바라고 있습니다.

## 라이선스 · 기여

MIT License. PR과 이슈는 언제나 환영합니다 — [GitHub](https://github.com/JunDamin/hwpapi)